<a href="https://colab.research.google.com/github/mayajdias/walmart-hurricane-project/blob/main/notebooks/Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning

## Decision Log

In [60]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

## Loading Data / EDA: distribution_centers.csv

In [56]:
import pandas as pd, numpy as np
distribution = pd.read_csv('distribution_centers.csv')
distribution.head()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


In [44]:
distribution.shape

(5, 8)

5 observations, 8 columns

In [45]:
distribution.dtypes

,0
dc_id,object
dc_name,object
city,object
state,object
avg_daily_shipments,int64
surge_capacity,object
distance_to_orlando_mi,int64
cost_per_shipment,float64


Need to change surge_capacity to int

In [46]:
distribution.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dc_id                   5 non-null      object 
 1   dc_name                 5 non-null      object 
 2   city                    5 non-null      object 
 3   state                   5 non-null      object 
 4   avg_daily_shipments     5 non-null      int64  
 5   surge_capacity          5 non-null      object 
 6   distance_to_orlando_mi  5 non-null      int64  
 7   cost_per_shipment       4 non-null      float64
dtypes: float64(1), int64(2), object(5)
memory usage: 452.0+ bytes


In [47]:
distribution.describe()

,avg_daily_shipments,distance_to_orlando_mi,cost_per_shipment
count,5.00000,5.000000,4.000000
mean,360.00000,485.000000,410.625000
std,96.17692,377.987434,197.064023
min,250.00000,141.000000,245.000000
25%,300.00000,235.000000,284.750000
50%,350.00000,438.000000,355.250000
75%,400.00000,503.000000,481.125000
max,500.00000,1108.000000,687.000000


In [48]:
distribution.isnull().sum()

,0
dc_id,0
dc_name,0
city,0
state,0
avg_daily_shipments,0
surge_capacity,0
distance_to_orlando_mi,0
cost_per_shipment,1


1 null item in cost_per_shipment

In [49]:
distribution["state"].value_counts()

,count
state,
FL,2
GA,1
TX,1
AL,1


Cleaning Required: remove non-numerical values from surge_capacity and make integer type. Address missing value in cost_per_shipment.

## Cleaning: distribution_centers.csv

In [57]:
def clean_distributions(df):
  df = df.copy()
  if "surge_capacity" in df.columns:
    df["surge_capacity"] = df["surge_capacity"].str.extract(r"(\d+)").astype(int)
  if "cost_per_shipment" in df.columns:
        mean_cost_per_shipment = df["cost_per_shipment"].mean()
        df["cost_per_shipment"] = df["cost_per_shipment"].fillna(mean_cost_per_shipment)
  return df

In [58]:
distributions_cleaned = clean_distributions(distribution)
distributions_cleaned.head()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.000
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.500
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.000
3,DC-MIA,Miami Distribution Center,Miami,FL,350,750,235,298.000
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,410.625


In [61]:
log("Cleaning", "converted surge_capacity to integer and addressed missing value in cost_per_shipment", "5")

[Cleaning] converted surge_capacity to integer and addressed missing value in cost_per_shipment (5 row(s))


## Loading Data / EDA: store_locations.csv

In [64]:
stores = pd.read_csv('store_locations.csv')
stores.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


In [15]:
stores.shape

(14, 8)

14 observations, 8 columns

In [18]:
stores.dtypes

,0
store_id,object
store_name,object
city,object
state,object
zip_code,int64
latitude,float64
longitude,float64
region,object


In [19]:
stores.duplicated().sum()

np.int64(0)

In [21]:
stores.duplicated(subset=["store_id"]).sum()

np.int64(0)

In [22]:
stores.duplicated(subset=["store_name"]).sum()

np.int64(2)

There are two duplicate store names

In [23]:
name_duplicates = stores[stores.duplicated(subset=["store_name"], keep=False)]
print(name_duplicates)

   store_id                             store_name       city    state  \
0    FL-239            Walmart Supercenter Orlando    Orlando       FL   
4    FL-287  Walmart Neighborhood Market Kissimmee  Kissimmee  Florida   
12    FL239            Walmart Supercenter Orlando    Orlando       FL   
13   FL_287  Walmart Neighborhood Market Kissimmee  Kissimmee  Florida   

    zip_code  latitude  longitude   region  
0      32801   28.5383   -81.3792  Central  
4      34741   28.2920   -81.4076  Central  
12     32801   28.5383   -81.3792  Central  
13     34741   28.2920   -81.4076  Central  


The two duplicated store names are not listed as total duplicates because store_id and state are formatted differently between them. Need to standardize format for these columns, then drop duplicate rows.

In [24]:
stores["city"].value_counts()

,count
city,
Orlando,2
Kissimmee,2
Miami,1
Tampa,1
Jacksonville,1
Daytona Beach,1
Fort Myers,1
Gainesville,1
Tallahassee,1


In [25]:
stores["state"].value_counts()

,count
state,
FL,10
Florida,3
Fla.,1


Need to standardize state name - all are in Florida, but formatted 3 ways. Should use "FL" because this is the convention in the previous data set.

In [26]:
stores["region"].value_counts()

,count
region,
Central,5
West,3
North,2
South,1
East,1
Panhandle,1
Panhandle,1


In [27]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    14 non-null     object 
 1   store_name  14 non-null     object 
 2   city        14 non-null     object 
 3   state       14 non-null     object 
 4   zip_code    14 non-null     int64  
 5   latitude    13 non-null     float64
 6   longitude   13 non-null     float64
 7   region      14 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 1.0+ KB


In [28]:
stores.describe()

,zip_code,latitude,longitude
count,14.000000,13.000000,13.000000
mean,33200.428571,28.418192,-82.069631
std,893.995759,1.319349,1.676108
min,32099.000000,25.761700,-87.216900
25%,32526.000000,27.950600,-82.324800
50%,32951.000000,28.292000,-81.655700
75%,33776.000000,29.210800,-81.379200
max,34741.000000,30.421300,-80.191800


In [29]:
stores.isnull().sum()

,0
store_id,0
store_name,0
city,0
state,0
zip_code,0
latitude,1
longitude,1
region,0


In [33]:
stores[stores['latitude'].isna()]

,store_id,store_name,city,state,zip_code,latitude,longitude,region
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle


In [34]:
stores[stores['longitude'].isna()]

,store_id,store_name,city,state,zip_code,latitude,longitude,region
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle


The missing values for latitude and longitude are both in the same row, the Walmart Supercenter Tallahassee.

Cleaning Required: standardize store_id format by removing filler characters ("-", "_", etc.). Standardize state name by setting all to "FL". Address missing latitude and longitude for Tallahassee store. Drop duplicate rows.

## Cleaning: store_locations.csv

In [67]:
def clean_stores(df):
  df = df.copy()
  if 'store_id' in df.columns:
        df['store_id'] = df['store_id'].astype(str).str.replace(r'[^a-zA-Z0-9]', '', regex=True)
  if 'state' in df.columns:
        df['state'] = 'FL'
  if 'latitude' in df.columns:
        tallahassee_latitude = 30.4383
        df['latitude'] = df['latitude'].fillna(tallahassee_latitude)
  if 'longitude' in df.columns:
        tallahassee_longitude = -84.2807
        df['longitude'] = df['longitude'].fillna(tallahassee_longitude)
  df = df.drop_duplicates()
  return df

In [68]:
stores_cleaned = clean_stores(stores)
stores_cleaned.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL287,Walmart Neighborhood Market Kissimmee,Kissimmee,FL,34741,28.2920,-81.4076,Central


In [70]:
log("Cleaning", "standardized store_id and state, filled missing latitude and longitude", "14")

[Cleaning] standardized store_id and state, filled missing latitude and longitude (14 row(s))


In [69]:
stores_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    12 non-null     object 
 1   store_name  12 non-null     object 
 2   city        12 non-null     object 
 3   state       12 non-null     object 
 4   zip_code    12 non-null     int64  
 5   latitude    12 non-null     float64
 6   longitude   12 non-null     float64
 7   region      12 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 864.0+ bytes


## Loading Data / EDA: walmart_transactions.csv

In [71]:
transactions = pd.read_csv("walmart_transactions.csv")
transactions.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


In [73]:
transactions.shape

(25562, 8)

25,562 observations, 8 columns

In [75]:
transactions.duplicated().sum()

np.int64(30)

30 duplicate rows even before cleaning

In [77]:
transactions.dtypes

,0
transaction_id,object
store_id,object
timestamp,object
sku,object
product_name,object
category,object
quantity,int64
unit_price,object


Need to make unit_price a float and timestamp a date

In [78]:
transactions["sku"].value_counts()

,count
sku,
TARP-HD,766
LED-FLASH,732
FLASH-01,728
HDTARP-01,701
CHRG-PORT,684
CANDLE-EMRG,664
CNDL-EM,652
BEER-24,643
FIRSTAID-KIT,637


Need to consolidate SKUs

In [79]:
transactions.isnull().sum()

,0
transaction_id,0
store_id,0
timestamp,886
sku,0
product_name,0
category,0
quantity,0
unit_price,0


timestamp is the only column with missing values, it has 886 missing values

In [80]:
DECISIONS

[{'step': 'Cleaning',
  'decision': 'converted surge_capacity to integer and addressed missing value in cost_per_shipment',
  'rows': '5'},
 {'step': 'Cleaning',
  'decision': 'standardized store_id and state, filled missing latitude and longitude',
  'rows': '14'}]